# 00b_setup_gold
Run this notebook ONCE before the first Gold load. It is safe to run again (everything uses IF NOT EXISTS).

It creates the Gold star schema:
* 4 dimensions: `dim_date`, `dim_hour`, `dim_event_type`, `dim_repo`
* 4 facts: `fact_activity_hourly`, `fact_daily_summary`, `fact_repo_daily`, `fact_repo_trend`
* `gold_aggregation_metrics` (rows aggregated and run time of every Gold load)

Every column type is written by hand. Keys are documented in the column comments (Delta does not enforce keys).
The larger tables use **Liquid Clustering** (`CLUSTER BY`) on the columns Power BI filters and joins on.

In [ ]:
%run ./common

In [ ]:
%run ./common_gold

In [ ]:
# Each entry: table name -> (column definitions, clustering columns or "" for none)
GOLD_DDL = {}

# ---------------- Dimensions ----------------
GOLD_DDL[DIM_DATE] = ("""
    date_key         INT       COMMENT 'PRIMARY KEY. yyyymmdd, e.g. 20261005',
    full_date        DATE,
    year             INT,
    month_num        INT,
    month_name       STRING,
    day_of_month     INT,
    day_of_week_num  INT       COMMENT '1 = Sunday ... 7 = Saturday',
    day_name         STRING,
    week_of_year     INT,
    is_weekend       BOOLEAN,
    hours_loaded     INT       COMMENT 'Hourly GH Archive files loaded for this day (0 to 24)',
    is_complete_day  BOOLEAN   COMMENT 'hours_loaded of at least COMPLETE_DAY_MIN_HOURS',
    load_timestamp   TIMESTAMP
""", "")

GOLD_DDL[DIM_HOUR] = ("""
    hour_key        INT       COMMENT 'PRIMARY KEY. Hour of day in UTC, 0 to 23',
    hour_label      STRING    COMMENT 'e.g. 09:00 UTC',
    hour_pkt        INT       COMMENT 'Same hour in Pakistan time (UTC+5)',
    day_part_pkt    STRING    COMMENT 'Night, Morning, Afternoon, Evening (Pakistan time)',
    load_timestamp  TIMESTAMP
""", "")

GOLD_DDL[DIM_EVENT_TYPE] = ("""
    event_type      STRING    COMMENT 'PRIMARY KEY. GitHub event type, e.g. PushEvent',
    event_category  STRING    COMMENT 'Code, Pull requests, Issues, Popularity, Releases, Repository structure, Other',
    load_timestamp  TIMESTAMP
""", "")

GOLD_DDL[DIM_REPO] = ("""
    repo_id           BIGINT    COMMENT 'PRIMARY KEY. Tracked repositories only (see common_gold rules)',
    repo_name         STRING    COMMENT 'owner/name, latest name seen',
    owner             STRING,
    repo_short_name   STRING,
    first_seen_date   DATE,
    last_seen_date    DATE,
    total_events      BIGINT    COMMENT 'Events in the whole loaded window',
    distinct_actors   BIGINT    COMMENT 'Different contributors in the whole loaded window',
    community_events  BIGINT    COMMENT 'Stars, forks, PRs, reviews, issues, comments, releases in the window',
    load_timestamp    TIMESTAMP
""", "repo_id")

# ---------------- Facts ----------------
GOLD_DDL[FACT_ACTIVITY_HOURLY] = ("""
    date_key         INT       COMMENT 'KEY part 1. FK to dim_date',
    hour_key         INT       COMMENT 'KEY part 2. FK to dim_hour',
    event_type       STRING    COMMENT 'KEY part 3. FK to dim_event_type',
    events           BIGINT,
    distinct_repos   BIGINT,
    distinct_actors  BIGINT,
    load_timestamp   TIMESTAMP
""", "date_key")

GOLD_DDL[FACT_DAILY_SUMMARY] = ("""
    date_key              INT     COMMENT 'PRIMARY KEY. FK to dim_date',
    events                BIGINT,
    active_repos          BIGINT  COMMENT 'All repositories with at least one event that day',
    active_actors         BIGINT  COMMENT 'Different contributors that day',
    active_tracked_repos  BIGINT  COMMENT 'Tracked repositories with at least one event that day',
    load_timestamp        TIMESTAMP
""", "")

GOLD_DDL[FACT_REPO_DAILY] = ("""
    date_key              INT     COMMENT 'KEY part 1. FK to dim_date',
    repo_id               BIGINT  COMMENT 'KEY part 2. FK to dim_repo',
    total_events          BIGINT,
    push_events           BIGINT,
    pull_request_events   BIGINT,
    pr_review_events      BIGINT  COMMENT 'Reviews and review comments',
    issue_events          BIGINT,
    issue_comment_events  BIGINT,
    release_events        BIGINT,
    watch_events          BIGINT  COMMENT 'Stars',
    fork_events           BIGINT,
    create_events         BIGINT,
    delete_events         BIGINT,
    distinct_actors       BIGINT,
    prs_opened            BIGINT,
    prs_closed            BIGINT,
    prs_merged            BIGINT  COMMENT 'action = merged (GH Archive reports merges as their own action)',
    issues_opened         BIGINT,
    issues_closed         BIGINT,
    load_timestamp        TIMESTAMP
""", "date_key, repo_id")

GOLD_DDL[FACT_REPO_TREND] = ("""
    date_key           INT      COMMENT 'KEY part 1. FK to dim_date',
    repo_id            BIGINT   COMMENT 'KEY part 2. FK to dim_repo',
    events             BIGINT   COMMENT 'Community events that day (PRs, reviews, issues, comments, stars, forks, releases; no push/create/delete)',
    has_full_baseline  BOOLEAN  COMMENT 'True when the 7 previous days are all complete days',
    baseline_avg       DOUBLE   COMMENT 'Average events per day over the previous 7 calendar days (quiet days count as 0)',
    baseline_std       DOUBLE   COMMENT 'Sample standard deviation over the previous 7 calendar days',
    z_score            DOUBLE   COMMENT '(events - baseline_avg) / baseline_std, only for repos with a baseline of at least 1 event per day',
    pct_vs_baseline    DOUBLE,
    events_last_7d     BIGINT   COMMENT 'This day and the 6 days before',
    events_prev_7d     BIGINT   COMMENT 'The 7 days before that',
    growth_pct_7d      DOUBLE   COMMENT 'Momentum: last 7 days against the previous 7 days, in percent',
    is_surge           BOOLEAN,
    surge_type         STRING   COMMENT 'Spike, Breakout or NULL',
    load_timestamp     TIMESTAMP
""", "date_key, repo_id")

# ---------------- Gold run metrics ----------------
GOLD_DDL[GOLD_METRICS] = ("""
    run_id            STRING,
    target_table      STRING,
    date_from         DATE,
    date_to           DATE,
    source_rows_read  BIGINT   COMMENT 'Silver rows aggregated',
    rows_written      BIGINT,
    start_time        TIMESTAMP,
    end_time          TIMESTAMP,
    duration_seconds  DOUBLE,
    load_timestamp    TIMESTAMP
""", "")

In [ ]:
# Create every table. If CLUSTER BY is refused for some reason, the table is created without it and the notebook continues.
for table_name in GOLD_DDL:
    columns_sql = GOLD_DDL[table_name][0]
    cluster_cols = GOLD_DDL[table_name][1]
    base_ddl = "CREATE TABLE IF NOT EXISTS " + table_name + " (" + columns_sql + ") USING DELTA"
    if cluster_cols != "":
        try:
            spark.sql(base_ddl + " CLUSTER BY (" + cluster_cols + ")")
            print("ready (clustered by " + cluster_cols + "):", table_name)
        except Exception as e:
            print("CLUSTER BY failed (" + str(e)[:150] + "). Creating without it:", table_name)
            spark.sql(base_ddl)
            print("ready (no clustering):", table_name)
    else:
        spark.sql(base_ddl)
        print("ready:", table_name)

In [ ]:
display(spark.sql("SHOW TABLES IN " + CATALOG + "." + SCHEMA))